# Parsing results and benchmarks

The measurements below cover the twelve schema 21 account runs. Each publication belongs to one run. The coverage denominator comes from the M1 publication–version inventory, and the handoff counts come from the M2 export manifests.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
from IPython.display import HTML, Markdown, display

data = Path("../../data_m2")
papers = pd.read_csv(data / "papers.csv", dtype={"arxiv_id": str}, keep_default_na=False)
versions = pd.read_csv(data / "versions.csv", dtype={"arxiv_id": str})
runs = pd.read_csv(data / "runs.csv")
kinds = pd.read_csv(data / "node_kinds.csv")
states = ["parsed", "partial", "empty_no_tex", "blocked_input", "failed"]
counts = versions.status.value_counts().reindex(states, fill_value=0)
assert papers.arxiv_id.is_unique and runs.run_id.is_unique
assert set(papers.run_id) == set(runs.run_id)
assert not versions.duplicated(["arxiv_id", "version"]).any()
by_paper = pd.crosstab(versions.arxiv_id, versions.status).reindex(columns=states, fill_value=0)
pd.testing.assert_frame_equal(by_paper, papers.set_index("arxiv_id")[states].sort_index(),
                              check_names=False, check_dtype=False)
assert kinds.occurrences.sum() == papers.elements_before_dedup.sum()

def table(frame):
    display(HTML(frame.to_html(index=False, border=0, classes="dataframe")))

def chart(fig, include_js=False):
    fig.update_layout(margin=dict(l=10, r=15, t=40, b=10))
    display(HTML(fig.to_html(full_html=False, include_plotlyjs=include_js,
                             config={"displaylogo": False, "responsive": True})))

## Publication completion and version coverage

A completed publication has a recorded task result. A clean parse requires the version status `parsed`. The runner's broader version coverage metric also counts `partial`, where some output exists but diagnostics remain.

In [ ]:
n_papers, n_versions = len(papers), len(versions)
ready = papers.loc[papers.export_ready]
metrics = [
    ("Publication tasks completed", runs.papers.sum(), n_papers),
    ("Publications eligible for handoff", len(ready), n_papers),
    ("Expected versions with a recorded outcome", papers.versions.sum(), n_versions),
    ("Cleanly parsed versions", counts["parsed"], n_versions),
    ("Versions yielding a full or partial parse", counts["parsed"] + counts["partial"], n_versions),
]
table(pd.DataFrame([(name, f"{num:,} / {den:,}", f"{num / den:.2%}") for name, num, den in metrics],
                   columns=["Measure", "Count / denominator", "Rate"]))
status_frame = counts.rename_axis("Version status").reset_index(name="Versions")
chart(px.bar(status_frame, x="Version status", y="Versions", text_auto=True,
             title="Outcome for every expected version"), include_js=True)

### Look up a publication

Enter a full arXiv ID to see the outcome of each version and the publication's handoff decision. The downloadable tables contain all {download}`publication measures <../../data_m2/papers.csv>` and {download}`version outcomes <../../data_m2/versions.csv>`.

In [ ]:
# Embed compact records so exact-ID lookup also works in a local HTML build.
lookup = {}
for paper in papers.itertuples(index=False):
    lookup[paper.arxiv_id] = [paper.exclusion_reason or "export_ready", []]
for row in versions.itertuples(index=False):
    lookup[row.arxiv_id][1].append(f"v{row.version}: {row.status}")

widget = """
<div id="m2-lookup">
  <label>arXiv ID <input type="search" value="2606.15734" maxlength="16"></label>
  <p aria-live="polite"></p>
</div>
<script>
(() => {
  const records = RECORDS;
  const box = document.getElementById("m2-lookup");
  const field = box.querySelector("input");
  const result = box.querySelector("p");
  function show() {
    const id = field.value.trim().replace("-", ".");
    const row = Object.hasOwn(records, id) && records[id];
    result.textContent = row ? id + " · " + row[1].join(", ") + " · " + row[0]
                             : "Enter an assigned arXiv ID, for example 2606.15734.";
  }
  field.addEventListener("input", show);
  show();
})();
</script>
"""
display(HTML(widget.replace("RECORDS", json.dumps(lookup, separators=(",", ":")))))

## Export decisions

The handoff writer assigns one primary reason per excluded publication. Unresolved citations take priority over a partial version, followed by a failed version. The remaining gate category records a publication that did not meet the other eligibility checks. The count of excluded publications and the count of affected versions therefore have different denominators.

In [ ]:
excluded = papers.loc[~papers.export_ready]
reasons = excluded.exclusion_reason.value_counts().rename_axis("Manifest reason").reset_index(name="Papers")
reasons["Share of excluded papers"] = reasons.Papers.map(lambda n: f"{n / len(excluded):.1%}")
table(reasons)
unresolved = excluded.exclusion_reason.eq("unresolved_citations").sum()
gate = excluded.loc[excluded.exclusion_reason.eq("gate_not_passed")]
blocked = versions.loc[versions.status.eq("blocked_input"), ["arxiv_id", "version"]]
m1 = pd.read_csv(data.parent / "data" / "versions.csv", dtype={"arxiv_id": str},
                 usecols=["arxiv_id", "version", "status"])
prior = blocked.merge(m1, on=["arxiv_id", "version"], how="left", validate="one_to_one")
assert prior.status.notna().all()
display(Markdown(f"Unresolved citations account for **{unresolved:,} of {len(excluded):,} exclusions**. "
                 f"**{gate.blocked_input.gt(0).sum()} of {len(gate)}** publications classified as "
                 f"`gate_not_passed` had at least one blocked input version. Across the full corpus, "
                 f"**{prior.status.eq('failed').sum()} of {len(blocked)}** M2 versions with "
                 f"`blocked_input` had status `failed` for the same publication and version in M1."))

## Elements and bibliography entries

Working counts include recoverable content from partial publications. The handoff contains the rows from export-ready publications. Element occurrences include roots and structural nodes alongside sentences and formulas. A hierarchy edge identifies a placement in one version.

In [ ]:
dedup = []
for label, prefix, exported in [("Elements", "elements", "elements_rows"),
                                ("BibTeX entries", "bibtex_entries", "bibtex_rows")]:
    before = papers[f"{prefix}_before_dedup"].sum()
    after = papers[f"{prefix}_after_dedup"].sum()
    published = runs[exported].sum()
    assert ready[f"{prefix}_after_dedup"].sum() == published
    dedup.append((label, f"{before:,}", f"{after:,}", f"{1 - after / before:.2%}", f"{published:,}"))
table(pd.DataFrame(dedup, columns=["Record", "Before dedup", "After dedup", "Reduction", "Handoff rows"]))
assert ready.hierarchy_edges.sum() == runs.hierarchy_rows.sum()
display(Markdown(f"The handoff contains **{runs.hierarchy_rows.sum():,}** version-specific hierarchy edges. "
                 f"Sentences account for **{kinds.set_index('kind').loc['sentence', 'occurrences'] / kinds.occurrences.sum():.1%}** "
                 "of node occurrences before deduplication."))
chart(px.bar(kinds.sort_values("occurrences"), x="occurrences", y="kind", orientation="h",
             height=480, labels={"kind":"Node kind", "occurrences":"Occurrences before deduplication"}))

## Parser performance on Colab

Each receipt measures the parse phase and the subsequent Parquet export. Wall time per publication divides their sum by the assigned publication count. Across accounts, adding wall times gives cumulative run time. The peak RSS sample sums the runner and its worker processes during parsing. The sampling window ends before export.

In [ ]:
totals = runs.total_wall_time_s.sum()
cpu_receipts = (runs.colab_cpu_verified & runs.environment.eq("colab_cpu")).sum()
display(Markdown(f"**{cpu_receipts} of {len(runs)} receipts** record Colab CPU. "
                 f"Cumulative parse and export time was **{totals:,.3f} s** "
                 f"(**{totals / n_papers:.3f} s per assigned publication**). "
                 f"The highest sampled process-tree RSS during parsing was **{runs.peak_ram_mb.max():,.2f} MiB**."))
run_table = pd.DataFrame({"Member": runs.member, "Account": runs.account, "Papers": runs.papers,
                          "Workers": runs.workers, "Parse (s)": runs.wall_time_s,
                          "Export (s)": runs.export_wall_time_s, "Total (s)": runs.total_wall_time_s,
                          "Total s/paper": (runs.total_wall_time_s / runs.papers).round(3),
                          "Peak RSS during parsing (MiB)": runs.peak_ram_mb})
table(run_table)
storage = [("Working files and intermediate artifacts", runs.working_bytes.sum()),
           ("Export directories, including manifests", runs.tables_directory_bytes.sum()),
           ("Parquet shards within those directories", runs.parquet_shard_bytes.sum())]
table(pd.DataFrame([(label, f"{value:,}", f"{value / (1024 ** 2):,.2f}") for label, value in storage],
                   columns=["Storage across 12 runs", "Bytes", "MiB"]))
display(Markdown("Shard storage includes all five handoff tables: papers and references from M1, "
                 "plus elements, hierarchy and bibtex from M2. Each run measures its own files."))